# 11.2 Components of Time Series

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.02_components-of-time-series.ipynb)

Companion code for Section 11.2 of the book. Run the cells in order from top to bottom.

**Example 11.2.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose, STL
import warnings; warnings.filterwarnings('ignore')

# ── Generate realistic time series with all components ────────────
np.random.seed(42)
n = 365 * 3   # 3 years daily data
dates = pd.date_range('2021-01-01', periods=n, freq='D')

# Trend: linear + slight quadratic
t = np.arange(n)
trend = 100 + 0.08*t + 0.000015*t**2

# Seasonality: annual (period=365) + weekly (period=7)
season_annual = 20 * np.sin(2 * np.pi * t / 365.25 - np.pi/2)
season_weekly = 8  * np.sin(2 * np.pi * t / 7)
seasonality   = season_annual + season_weekly

# Irregular noise + occasional spikes
noise = np.random.normal(0, 5, n)
spikes = np.zeros(n)
spike_idx = np.random.choice(n, 15, replace=False)
spikes[spike_idx] = np.random.uniform(-25, 35, 15)

# Compose series
y = trend + seasonality + noise + spikes
ts = pd.Series(y, index=dates, name='value')

# ── Classical additive decomposition ─────────────────────────────
decomp = seasonal_decompose(ts, model='additive', period=365, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('Time Series Decomposition - Additive Model (3 Years Daily)',
            fontsize=14, fontweight='bold')

components = [
    ('Original', ts,              '#1F3864', 'Observed values'),
    ('Trend',    decomp.trend,    '#2E75B6', 'Long-run level change'),
    ('Seasonal', decomp.seasonal, '#43A047', 'Periodic fluctuation (period=365)'),
    ('Residual', decomp.resid,    '#E53935', 'Unexplained noise'),
]

for ax, (label, data, color, desc) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, lw=1.5 if label!='Original' else 1.0,
           alpha=0.9)
    ax.set_ylabel(label, fontsize=11, fontweight='bold')
    ax.set_title(f'{label}: {desc}', fontsize=10)
    ax.grid(alpha=0.3)
    if label == 'Residual':
        ax.axhline(0, color='black', lw=1, ls='--')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('ts_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

# ── STL Decomposition (more robust) ──────────────────────────────
stl = STL(ts, period=365, robust=True)
stl_result = stl.fit()

print('=== STL Decomposition Summary ===')
print(f'Trend range:    [{stl_result.trend.min():.1f}, {stl_result.trend.max():.1f}]')
print(f'Seasonal range: [{stl_result.seasonal.min():.1f}, {stl_result.seasonal.max():.1f}]')
print(f'Residual std:   {stl_result.resid.std():.4f}')

# Strength of trend and seasonality
var_total    = ts.var()
var_residual = stl_result.resid.var()
var_trend    = stl_result.trend.var()
var_season   = stl_result.seasonal.var()

strength_trend  = max(0, 1 - var_residual / (var_residual + var_trend))
strength_season = max(0, 1 - var_residual / (var_residual + var_season))
print(f'\nTrend strength:     {strength_trend:.4f}  (0=no trend, 1=pure trend)')
print(f'Seasonal strength:  {strength_season:.4f}  (0=no seasonality, 1=pure seasonality)')

# Component contribution
print(f'\nVariance contributions:')
print(f'  Trend:      {var_trend/var_total:.1%} of total variance')
print(f'  Seasonality:{var_season/var_total:.1%} of total variance')
print(f'  Residual:   {var_residual/var_total:.1%} of total variance')